# Phase 3: Automated Technical Sandbox Evaluation

This notebook extracts generated string data from the Phase 2 output corpus and executes automated test verification routines inside a local Linux/WSL sandbox environment.

## 1. Pre-Processing & Extraction
* Removes generative reasoning meta-tags (`<think>...</think>`).
* Isolates clean C code blocks and parses machine-readable JSON testing manifested suites.

## 2. Static Compilation Analysis
* Compiles extracted code loops using `gcc` (v11.4.0).

## 3. Dynamic Runtime Instrumentations
* **Valgrind Triage:** Monitors memory allocations to detect active leaks, invalid pointer reads/writes, or dynamic vulnerabilities.
* **UBSan Instrumentation:** Re-compiles code using undefined behavior sanitizers (`-fsanitize=undefined`) to capture silent overflow overflows.
* **Triage Logging:** Classifies outputs into Clean Pass, Safety Violation, Infinite Loop Timeout, or Compilation Error.

In [ ]:
import pandas as pd
import os
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/LLM_Domain_Results/Supplementary Datasets/'

csv_files = [
    'GPTOSS_Phase2_External_Audit_Results.csv',
    'LLAMA_Phase2_External_Audit_Results.csv',
    'KIMIK2_Phase2_External_Audit_Results.csv'
]

dfs = []
for file in csv_files:
    file_path = os.path.join(BASE_DIR, file)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        # Ensure text columns are strings to prevent NaN regex crashes
        df['step2_response'] = df['step2_response'].fillna('').astype(str)
        df['step6_response'] = df['step6_response'].fillna('').astype(str)
        dfs.append(df)
        print(f"Loaded {len(df)} rows from {file}")
    else:
        print(f"Warning: Could not find {file}")

# Combine into one master external dataset
master_ext_df = pd.concat(dfs, ignore_index=True)
master_ext_df = master_ext_df[~master_ext_df['global_problem_id'].str.contains('MD_', case=False, na=False)]

print("\n==================================================")
print(f"PHASE 3B QUEUE READY: {len(master_ext_df)} Total Runs")
print("==================================================")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 2839 rows from GPTOSS_Phase2_External_Audit_Results.csv
Loaded 2892 rows from LLAMA_Phase2_External_Audit_Results.csv
Loaded 2892 rows from KIMIK2_Phase2_External_Audit_Results.csv

PHASE 3B QUEUE READY: 8551 Total Runs


In [ ]:
import re
import json

def clean_think_tags(text):
    """Removes latent <think>...</think> blocks from the response."""
    if not isinstance(text, str):
        return ""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE).strip()

def extract_c_code(response_text):
    """Extracts C code from markdown blocks or returns raw text if no block exists."""
    cleaned_text = clean_think_tags(response_text)

    # 1. Try explicit C code fence
    matches = re.findall(r'```c(.*?)```', cleaned_text, re.DOTALL | re.IGNORECASE)
    if matches:
        return matches[0].strip()

    # 2. Try generic code fence
    matches = re.findall(r'```(.*?)```', cleaned_text, re.DOTALL)
    if matches:
        return matches[0].strip()

    # 3. Fallback to raw text
    return cleaned_text.strip()

def extract_json_test_cases(response_text):
    """Finds and parses the JSON test case block."""
    cleaned_text = clean_think_tags(response_text)
    if not cleaned_text:
        return None

    # 1. Try fenced JSON block first
    fence_match = re.search(r'```json(.*?)```', cleaned_text, re.DOTALL | re.IGNORECASE)
    if fence_match:
        candidate = fence_match.group(1).strip()
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass  # fall through

    # 2. Fallback: try to grab the first JSON-looking object
    # Using greedy match \{.*\} because test cases contain nested braces (e.g., list of dicts)
    brace_match = re.search(r'\{.*\}', cleaned_text, re.DOTALL)
    if brace_match:
        candidate = brace_match.group(0).strip()
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            return None

    return None

# Apply extractions directly to the DataFrame
print("Extracting C code and Test Cases...")
master_ext_df['extracted_code'] = master_ext_df['step2_response'].apply(extract_c_code)
master_ext_df['extracted_tests'] = master_ext_df['step6_response'].apply(extract_json_test_cases)

# Calculate failures
empty_code_mask = master_ext_df['extracted_code'] == ''
missing_tests_mask = master_ext_df['extracted_tests'].isna()

print("\n--- EXTRACTION COMPLETE ---")
print(f"Rows missing code: {empty_code_mask.sum()} / {len(master_ext_df)}")
print(f"Rows missing/invalid JSON tests: {missing_tests_mask.sum()} / {len(master_ext_df)}")

# Print debugging info for failures (if any)
if empty_code_mask.sum() > 0:
    print("\n--- SAMPLE ROWS WITH MISSING CODE ---")
    print(master_ext_df.loc[empty_code_mask, ['global_problem_id', 'model_name']].head())

if missing_tests_mask.sum() > 0:
    print("\n--- SAMPLE ROWS WITH INVALID JSON ---")
    print(master_ext_df.loc[missing_tests_mask, ['global_problem_id', 'model_name']].head())

Extracting C code and Test Cases...

--- EXTRACTION COMPLETE ---
Rows missing code: 0 / 8551
Rows missing/invalid JSON tests: 187 / 8551

--- SAMPLE ROWS WITH INVALID JSON ---
    global_problem_id           model_name
7              LC_647  openai/gpt-oss-120b
55            LC_1263  openai/gpt-oss-120b
84            LC_2714  openai/gpt-oss-120b
124           LC_1003  openai/gpt-oss-120b
240           LC_2416  openai/gpt-oss-120b


In [ ]:
import subprocess
import tempfile
import os

def compile_code(code_string, compiler='gcc', extra_flags=None):
    if extra_flags is None:
        extra_flags = ['-O2', '-Wall', '-std=c11']

    with tempfile.NamedTemporaryFile(suffix=".c", delete=False, mode='w', encoding='utf-8') as f:
        f.write(code_string)
        src_path = f.name

    out_path = src_path[:-2] + ".out"
    cmd = [compiler, src_path, '-o', out_path] + extra_flags

    result = subprocess.run(cmd, capture_output=True, text=True, errors='replace')

    if result.returncode == 0:
        return True, out_path, result.stderr
    else:
        if os.path.exists(src_path): os.remove(src_path)
        return False, None, result.stderr

def run_valgrind(binary_path, test_data):
    if not test_data or 'test_suite' not in test_data:
        return "No Tests Available"

    if not isinstance(test_data['test_suite'], list):
        return "No Tests Available"

    valgrind_cmd = ['valgrind', '--leak-check=full', '--track-origins=yes', '--error-exitcode=1', binary_path]

    exit_val = test_data.get('exit_command', '')
    exit_command = str(exit_val) if exit_val is not None else ""

    for test in test_data['test_suite']:
        if not isinstance(test, dict):
            continue

        input_val = test.get('input', '')
        input_str = str(input_val) if input_val is not None else ""

        if exit_command:
            input_str += f"\n{exit_command}\n"

        try:
            result = subprocess.run(valgrind_cmd, input=input_str, capture_output=True, text=True, timeout=10, errors='replace')
            if result.returncode != 0:
                return "Memory Leak/Error Detected"
        except subprocess.TimeoutExpired:
            return "Timeout"

    return "Pass"

def run_ubsan(binary_path, test_data):
    if not test_data or 'test_suite' not in test_data:
        return "No Tests Available"

    if not isinstance(test_data['test_suite'], list):
        return "No Tests Available"

    exit_val = test_data.get('exit_command', '')
    exit_command = str(exit_val) if exit_val is not None else ""

    for test in test_data['test_suite']:
        if not isinstance(test, dict):
            continue

        input_val = test.get('input', '')
        input_str = str(input_val) if input_val is not None else ""

        if exit_command:
            input_str += f"\n{exit_command}\n"

        try:
            result = subprocess.run([binary_path], input=input_str, capture_output=True, text=True, timeout=10, errors='replace')
            if "runtime error:" in result.stderr:
                return "Undefined Behavior Detected"
        except subprocess.TimeoutExpired:
            return "Timeout"

    return "Pass"

print("Sandbox functions loaded successfully with null and garbage-byte safety.")

Sandbox functions loaded successfully with null and garbage-byte safety.


In [ ]:
print("Verifying Valgrind Installation...")
!apt-get update -qq && apt-get install -y valgrind -qq

Verifying Valgrind Installation...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
from tqdm.auto import tqdm
import os
import pandas as pd

OUTPUT_PATH = os.path.join(BASE_DIR, 'Phase3_External_TechEval_Results.csv')
SAVE_EVERY_N = 50

# --- CHECKPOINT RESUME LOGIC ---
completed_tasks = set()
if os.path.exists(OUTPUT_PATH):
    try:
        existing_df = pd.read_csv(OUTPUT_PATH)
        for _, row in existing_df.iterrows():
            completed_tasks.add((row['global_problem_id'], row['model_name']))
        print(f"Resuming from checkpoint: Found {len(completed_tasks)} completed runs.")
    except pd.errors.EmptyDataError:
        print("Existing CSV is empty. Starting fresh.")
else:
    print("No existing checkpoint found. Starting fresh.")

batch_results = []
runs_since_last_save = 0
total_runs = len(master_ext_df)


stats = {
    "processed": len(completed_tasks),
    "clean": 0,
    "safety_violation": 0,
    "timeout": 0,
    "error": 0,
    "no_tests": 0,
    "compiled": 0,
    "failed_compile": 0,
    "vg_pass": 0,
    "ub_pass": 0
}

print("\n" + "=" * 55)
print("STARTING TECHNICAL EVALUATION PIPELINE (PHASE 3B)")
print("=" * 55)

# Calculate remaining runs for the progress bar
remaining_runs = total_runs - len(completed_tasks)
progress_bar = tqdm(total=remaining_runs, desc="Auditing Binaries", unit="run")

for idx, row in master_ext_df.iterrows():
    prob_id = row['global_problem_id']
    current_model = row['model_name']

    # 1. THE CHECKPOINT SKIP
    if (prob_id, current_model) in completed_tasks:
        continue

    code = (row['extracted_code'] or '').strip()
    tests = row['extracted_tests']

    comp_status = "Fail"
    csr_binary = 0
    valgrind_status = "Skipped (Did Not Compile)"
    ubsan_status = "Skipped (Did Not Compile)"
    tests_available = 0

    if tests and 'test_suite' in tests and isinstance(tests['test_suite'], list):
        tests_available = 1
    else:
        stats["no_tests"] += 1

    if code:
        success, bin_path, err = compile_code(code)
        if success:
            comp_status = "Pass"
            csr_binary = 1
            stats["compiled"] += 1

            valgrind_status = run_valgrind(bin_path, tests)
            if valgrind_status == "Pass":
                stats["vg_pass"] += 1

            if os.path.exists(bin_path): os.remove(bin_path)

            ub_flags = ['-std=c11', '-O0', '-Wall', '-fsanitize=undefined', '-g']
            ub_success, ub_bin_path, ub_err = compile_code(code, extra_flags=ub_flags)

            if ub_success:
                ubsan_status = run_ubsan(ub_bin_path, tests)
                if ubsan_status == "Pass":
                    stats["ub_pass"] += 1
                if os.path.exists(ub_bin_path): os.remove(ub_bin_path)
            else:
                ubsan_status = "UBSan Compile Failed"

            # Classify the execution result based on Phase 03 definitions
            if valgrind_status == "Pass" and ubsan_status == "Pass":
                stats["clean"] += 1
            elif valgrind_status == "Timeout" or ubsan_status == "Timeout":
                stats["timeout"] += 1
            elif valgrind_status == "Memory Leak/Error Detected" or ubsan_status == "Undefined Behavior Detected":
                stats["safety_violation"] += 1
            elif "Failed" in ubsan_status or "Available" in valgrind_status:
                stats["error"] += 1
        else:
            stats["failed_compile"] += 1
            stats["error"] += 1 # Compilation failure counts as an error
    else:
        stats["failed_compile"] += 1
        stats["error"] += 1

    # Stage row for saving
    row_data = {
        'global_problem_id': prob_id,
        'source': row['source'],
        'model_name': current_model,
        'CompStatus': comp_status,
        'CSRBinary': csr_binary,
        'TestsAvailable': tests_available,
        'ValgrindStatus': valgrind_status,
        'UBStatus': ubsan_status
    }

    batch_results.append(row_data)
    completed_tasks.add((prob_id, current_model))
    runs_since_last_save += 1
    stats["processed"] += 1
    progress_bar.update(1)

    # --- CHECKPOINT SAVE ---
    if runs_since_last_save >= SAVE_EVERY_N:
        batch_df = pd.DataFrame(batch_results)
        write_header = not os.path.exists(OUTPUT_PATH)
        batch_df.to_csv(OUTPUT_PATH, mode='a', header=write_header, index=False)
        batch_results = []
        runs_since_last_save = 0

    if stats["processed"] % 10 == 0:
        progress_bar.set_postfix(clean=stats["clean"], fails=stats["safety_violation"], hangs=stats["timeout"])

progress_bar.close()

# --- FINAL FLUSH ---
if batch_results:
    batch_df = pd.DataFrame(batch_results)
    write_header = not os.path.exists(OUTPUT_PATH)
    batch_df.to_csv(OUTPUT_PATH, mode='a', header=write_header, index=False)

# Reload the full CSV to compute final metrics accurately across all batches
final_eval_df = pd.read_csv(OUTPUT_PATH)

# =====================================================================
# THE COMBINED FINAL PRINTOUT
# =====================================================================
print("\n" + "=" * 55)
print("FULL DYNAMIC ANALYSIS COMPLETE")
print("=" * 55)

print("--- High-Level Phase 03 Parity ---")
print(f"Clean Executions:      {stats['clean']}")
print(f"Safety Violations:     {stats['safety_violation']} (Leaks/UB)")
print(f"Logical Hangs:         {stats['timeout']} (Infinite Loops)")
print(f"System Errors:         {stats['error']} (Compile/Test Fails)")

print("\n--- Granular Technical Breakdown ---")
print(f"Successful Compiles:   {stats['compiled']}")
print(f"Failed Compiles:       {stats['failed_compile']}")
print(f"Valgrind Clean Passes: {stats['vg_pass']}")
print(f"UBSan Clean Passes:    {stats['ub_pass']}")
print(f"Invalid/Missing Tests: {stats['no_tests']}")

print("-" * 55)
print(f"Total Generations Processed: {stats['processed']} / {total_runs}")
print("=" * 55)

# Display a quick summary for your supervisory meeting!
print("\nQuick Summary (Compile Success Rate by Model):")
print(final_eval_df.groupby('model_name')['CSRBinary'].mean().round(3) * 100)

Resuming from checkpoint: Found 6635 completed runs.

STARTING TECHNICAL EVALUATION PIPELINE (PHASE 3B)


Auditing Binaries:   0%|          | 0/1916 [00:00<?, ?run/s]


FULL DYNAMIC ANALYSIS COMPLETE
--- High-Level Phase 03 Parity ---
Clean Executions:      370
Safety Violations:     1168 (Leaks/UB)
Logical Hangs:         8 (Infinite Loops)
System Errors:         442 (Compile/Test Fails)

--- Granular Technical Breakdown ---
Successful Compiles:   1573
Failed Compiles:       415
Valgrind Clean Passes: 380
UBSan Clean Passes:    1503
Invalid/Missing Tests: 34
-------------------------------------------------------
Total Generations Processed: 8623 / 8551

Quick Summary (Compile Success Rate by Model):
model_name
meta-llama/llama-3.3-70b-instruct    92.1
moonshotai/kimi-k2-0905              50.2
openai/gpt-oss-120b                  72.0
Name: CSRBinary, dtype: float64


In [ ]:
import pandas as pd
import os

# Load the final dataset
OUTPUT_PATH = os.path.join(BASE_DIR, 'Phase3_External_TechEval_Results.csv')
final_eval_df = pd.read_csv(OUTPUT_PATH)

# Calculate Granular Stats
total_processed = len(final_eval_df)
compiled = final_eval_df['CSRBinary'].sum()
failed_compile = total_processed - compiled
no_tests = (final_eval_df['TestsAvailable'] == 0).sum()
vg_pass = (final_eval_df['ValgrindStatus'] == "Pass").sum()
ub_pass = (final_eval_df['UBStatus'] == "Pass").sum()

# Calculate Phase 03 Parity Stats
clean = ((final_eval_df['ValgrindStatus'] == "Pass") & (final_eval_df['UBStatus'] == "Pass")).sum()
timeout = ((final_eval_df['ValgrindStatus'] == "Timeout") | (final_eval_df['UBStatus'] == "Timeout")).sum()
safety_violation = ((final_eval_df['ValgrindStatus'] == "Memory Leak/Error Detected") | (final_eval_df['UBStatus'] == "Undefined Behavior Detected")).sum()

# Any row that isn't a Clean Pass, a Timeout, or a Safety Violation is a System Error (Compile Fails, Missing Tests, etc.)
error = total_processed - clean - timeout - safety_violation

# =====================================================================
# THE COMBINED FINAL PRINTOUT
# =====================================================================
print("\n" + "=" * 55)
print("FULL DYNAMIC ANALYSIS COMPLETE (ALL RUNS)")
print("=" * 55)

print("--- High-Level Phase 03 Parity ---")
print(f"Clean Executions:      {clean}")
print(f"Safety Violations:     {safety_violation} (Leaks/UB)")
print(f"Logical Hangs:         {timeout} (Infinite Loops)")
print(f"System Errors:         {error} (Compile/Test Fails)")

print("\n--- Granular Technical Breakdown ---")
print(f"Successful Compiles:   {compiled}")
print(f"Failed Compiles:       {failed_compile}")
print(f"Valgrind Clean Passes: {vg_pass}")
print(f"UBSan Clean Passes:    {ub_pass}")
print(f"Invalid/Missing Tests: {no_tests}")

print("-" * 55)
print(f"Total Generations Processed: {total_processed} / {total_processed}")
print("=" * 55)

print("\nQuick Summary (Compile Success Rate by Model):")
print(final_eval_df.groupby('model_name')['CSRBinary'].mean().round(3) * 100)


FULL DYNAMIC ANALYSIS COMPLETE (ALL RUNS)
--- High-Level Phase 03 Parity ---
Clean Executions:      3182
Safety Violations:     2768 (Leaks/UB)
Logical Hangs:         96 (Infinite Loops)
System Errors:         2577 (Compile/Test Fails)

--- Granular Technical Breakdown ---
Successful Compiles:   6161
Failed Compiles:       2462
Valgrind Clean Passes: 3216
UBSan Clean Passes:    5749
Invalid/Missing Tests: 194
-------------------------------------------------------
Total Generations Processed: 8623 / 8623

Quick Summary (Compile Success Rate by Model):
model_name
meta-llama/llama-3.3-70b-instruct    92.1
moonshotai/kimi-k2-0905              50.2
openai/gpt-oss-120b                  72.0
Name: CSRBinary, dtype: float64
